# VCC 2026 submission: public-screen effect borrowing

Builds the validation-phase submission (contexts A, B, C) from the method in `models/public_screens/` (RESULTS.md Round 4; preset c1_wide by default: the per-target predictions of c1_calibrated plus the cross-line shared knockdown response called on ~8,000 genes, which fixes the -1.26 direction-fidelity score of the first submission; c4_wide also amplifies the per-target part 4x) and submits it with the official `vcc` CLI.

* Each of the 300 targets gets 400 predicted cells per context: real control cells from that context with their counts scaled gene by gene by the predicted fold change (binomial thinning down, added Poisson counts up), so values stay raw integer counts as the 2026 scorer requires.
* Predicted change = H1 shared response + similarity-weighted H1 effects of other genes (similarity from the Replogle 2022 K562 genome-wide screen) + the target's own K562 profile, gated to genes detected in >= 1% of the context's controls; the target drops to its K562 remaining fraction.
* `vcc prep` needs ~25 GB RAM for a full submission, so `package_vcc.py` runs the official prep check per context and on the combined labels, then packages the `.vcc` exactly as prep does. On a High-RAM runtime you can run plain `vcc prep` instead (last cell).

The API key is asked for at runtime and only kept in this runtime's environment. Nothing is written to Drive or the repo.

Runtime: standard CPU, ~40 GB free disk; about 20 minutes to build plus upload time.

In [ ]:
!pip -q install anndata h5py scipy pandas
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!uv tool install vcc-cli && vcc --version
# the wide-shared-response branch until its PR is merged, then main
!git clone -q --depth 1 -b wide-shared-response https://github.com/KunaalAgarwal/virtual_cell_notebooks /content/vcn || git clone -q --depth 1 https://github.com/KunaalAgarwal/virtual_cell_notebooks /content/vcn
WORK = '/content/vcc2026'; os.makedirs(WORK, exist_ok=True); os.chdir(WORK)

## API key (asked once, kept only in this runtime)

In [ ]:
import getpass
os.environ['VCC_TOKEN'] = getpass.getpass('VCC API key: ')
!vcc whoami

## Data: 2026 controls bundle and the Replogle K562 genome-wide screen

In [ ]:
!vcc datasets download controls -o {WORK}/vcc_2026_controls.zip
!unzip -o -j -q {WORK}/vcc_2026_controls.zip -d {WORK}/data
!ls -la {WORK}/data
!wget -q -O {WORK}/K562_gwps_normalized_bulk_01.h5ad https://ndownloader.figshare.com/files/35773217

## Build the prediction (streams to disk; ~3 GB RAM)

In [ ]:
M = '/content/vcn/models'
PRESET = 'c1_wide'   # second file of the day: 'c4_wide'
!python {M}/submission_2026/build_submission.py --data {WORK}/data \
    --h1 {M}/submission_2026/data/h1_pseudobulk.npz --k562 {WORK}/K562_gwps_normalized_bulk_01.h5ad \
    --method-dir {M}/public_screens --out {WORK}/pred.h5ad --diag {WORK}/diag --preset {PRESET}

## Validate with the official CLI and package the .vcc

In [ ]:
PY = os.path.expanduser('~/.local/share/uv/tools/vcc-cli/bin/python')
!{PY} {M}/submission_2026/package_vcc.py --pred {WORK}/pred.h5ad --data {WORK}/data --out {WORK}/pred.vcc --tmp {WORK}

## Submit (counts toward the daily limit of two)

In [ ]:
!vcc submit {WORK}/pred.vcc -m "screen-borrow {PRESET}" --wait

### Optional, High-RAM runtime only: the one-step official prep instead of `package_vcc.py`

In [ ]:
# !vcc prep {WORK}/pred.h5ad -g {WORK}/data/gene_names.csv --perts {WORK}/data/pert_counts.csv -o {WORK}/pred_official.vcc